# Peyton — Phase 2: Notes and Note Request cleaning

**Deliverables:** two cleaned datasets, reproducible cleaning code, and an audit report.

**Agreed inputs:** `notes-00001.tsv` and `batSignals-00000.tsv`. This is separate from Module 6 Ratings work. No Ratings or Status History input is needed.

Run cells in order. The user confirmed notes-00001.tsv as the Notes input. Original Desktop ZIP archives were extracted into data/raw without modifying the archives. The script contains the cleaning implementation; this notebook explains and runs it.

## 1. Configure inputs and scope

Start with `PREVIEW_ROWS = 1000` if desired, then use `None` for the final full-file run. Preview output goes into a separate subfolder. Do not infer whole-file relationships from a preview.

Supply `CUTOFF` only if the team knows the observation cutoff, including timezone. `SNAPSHOT_LABEL` records the actual source version/download information; file modification time is not a substitute.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

# Supports starting Jupyter from this folder, the project, or the workspace.
candidates = [Path.cwd(), Path.cwd() / 'phase2-peyton',
              Path.cwd() / 'misinformation-analysis' / 'phase2-peyton']
WORKFLOW = next((p for p in candidates if (p / 'clean_notes_requests.py').is_file()), None)
if WORKFLOW is None:
    raise FileNotFoundError('Open this notebook from the project or phase2-peyton folder.')
sys.path.insert(0, str(WORKFLOW.resolve()))
from clean_notes_requests import run, load_source, profile, REQUEST_TIMES

NOTES_PATH = WORKFLOW / 'data' / 'raw' / 'notes-00001.tsv'
REQUESTS_PATH = WORKFLOW / 'data' / 'raw' / 'batSignals-00000.tsv'
OUTPUT_DIR = WORKFLOW / 'outputs'
PREVIEW_ROWS = None  # None = every row in the agreed files; 1000 = preview only
CUTOFF = None  # Example format only: '2026-10-15T00:00:00Z'; use a verified cutoff
SNAPSHOT_LABEL = None

availability = pd.DataFrame([
    {'dataset': name, 'path': str(path), 'available': path.is_file()}
    for name, path in [('Notes', NOTES_PATH), ('Note Request', REQUESTS_PATH)]
])
display(availability)
if not availability.available.all():
    print('INPUTS MISSING: update the paths before running the remaining cells.')

,dataset,path,available
0,Notes,/Users/peytontom/dso576/misinformation-analysi...,True
1,Note Request,/Users/peytontom/dso576/misinformation-analysi...,True


## 2. Inspect the structure before cleaning

This cell reads only the first 1,000 rows of each file for an inexpensive initial inspection. These are the first rows, not a random sample. The cleaning cell below separately reads the configured scope.

Inspect IDs, timestamp values, optional text, and extra columns. Missing required columns stop the workflow instead of silently being ignored.

In [2]:
for name, path, required in [
    ('notes', NOTES_PATH, ['noteId', 'tweetId', 'createdAtMillis']),
    ('requests', REQUESTS_PATH, ['tweetId'] + REQUEST_TIMES),
]:
    sample = load_source(path, required, preview_rows=1000)
    print(f'{name}: first {len(sample):,} rows; {len(sample.columns)} columns')
    display(sample.head())
    display(profile(sample, name, 'initial sample'))

notes: first 1,000 rows; 24 columns


,noteId,noteAuthorParticipantId,createdAtMillis,tweetId,classification,believable,harmful,validationDifficulty,misleadingOther,misleadingFactualError,...,misleadingSatire,notMisleadingOther,notMisleadingFactuallyCorrect,notMisleadingOutdatedButNotWhenWritten,notMisleadingClearlySatire,notMisleadingPersonalOpinion,trustworthySources,summary,isMediaNote,isCollaborativeNote
0,2076363424486076857,F75B04680623CD9A452496D4AB69A20ED730CE93652C71...,1783878697379,17453093548,NOT_MISLEADING,,,,0,0,...,0,0,0,0,0,0,0,This post dates to July 2010. It promoted the ...,0,1
1,2096040661472469421,F75B04680623CD9A452496D4AB69A20ED730CE93652C71...,1788570144074,255751329392234496,MISINFORMED_OR_POTENTIALLY_MISLEADING,,,,0,0,...,0,0,0,0,0,0,0,This was the attendance for Manchester City's ...,0,1
2,2109518354130026772,F75B04680623CD9A452496D4AB69A20ED730CE93652C71...,1791783408840,269190965758156800,MISINFORMED_OR_POTENTIALLY_MISLEADING,,,,0,0,...,0,0,0,0,0,0,0,"This post is from Nov 15, 2012, during Michael...",0,1
3,2082124719525130520,F75B04680623CD9A452496D4AB69A20ED730CE93652C71...,1785252350875,1253353708961902596,MISINFORMED_OR_POTENTIALLY_MISLEADING,,,,0,0,...,0,0,0,0,0,0,0,"US intel agencies (FBI moderate conf., DOE/CIA...",0,1
4,2062018081854202070,F75B04680623CD9A452496D4AB69A20ED730CE93652C71...,1780458426732,1286632257076289536,NOT_MISLEADING,,,,0,0,...,0,0,0,0,0,0,0,The Tampa Bay Rays posted on July 24 2020: Tod...,0,1


,dataset,stage,column,dtype,rows,empty,whitespace_only,blank_total
0,notes,initial sample,noteId,string,1000,0,0,0
1,notes,initial sample,noteAuthorParticipantId,string,1000,0,0,0
2,notes,initial sample,createdAtMillis,string,1000,0,0,0
3,notes,initial sample,tweetId,string,1000,0,0,0
4,notes,initial sample,classification,string,1000,0,0,0
5,notes,initial sample,believable,string,1000,1000,0,1000
6,notes,initial sample,harmful,string,1000,1000,0,1000
7,notes,initial sample,validationDifficulty,string,1000,1000,0,1000
8,notes,initial sample,misleadingOther,string,1000,0,0,0
9,notes,initial sample,misleadingFactualError,string,1000,0,0,0


requests: first 1,000 rows; 7 columns


,tweetId,noteRequestFeedEligibleAtMillis,apiSmallFeedEligibleAtMillis,apiLargeFeedEligibleAtMillis,apiXlFeedEligibleAtMillis,sourceLinks,suggestions
0,1463732880,-1,-1,-1,1787257421863,,[]
1,1711529307,-1,-1,-1,1788047004994,,[]
2,2095550192,-1,-1,-1,1790712587377,,"[{""suggestion_id"":2105026835134218452,""suggest..."
3,4176682202,-1,-1,-1,1786803953724,,"[{""suggestion_id"":2088632163939832178,""suggest..."
4,6898415438,-1,-1,-1,1790006433504,,[]


,dataset,stage,column,dtype,rows,empty,whitespace_only,blank_total
0,requests,initial sample,tweetId,string,1000,0,0,0
1,requests,initial sample,noteRequestFeedEligibleAtMillis,string,1000,0,0,0
2,requests,initial sample,apiSmallFeedEligibleAtMillis,string,1000,0,0,0
3,requests,initial sample,apiLargeFeedEligibleAtMillis,string,1000,0,0,0
4,requests,initial sample,apiXlFeedEligibleAtMillis,string,1000,0,0,0
5,requests,initial sample,sourceLinks,string,1000,836,0,836
6,requests,initial sample,suggestions,string,1000,0,0,0


## 3. Cleaning rules

1. Preserve source files and all original field values. IDs remain strings.
2. Count empty and whitespace-only fields separately. Do not invent missing IDs, dates, or text.
3. Remove only exact duplicates across all source columns. Retain conflicting repeated `noteId` values in Notes and repeated `tweetId` values in Note Request, with flags.
4. Derive UTC timestamps from milliseconds. Flag missing, malformed, out-of-range, and nonpositive values. Only check dates beyond a cutoff when a verified cutoff is supplied.
5. Treat multiple notes for one tweet as a valid relationship. Compare tweet membership using aggregated counts, avoiding row multiplication.
6. Retain unmatched tweets and all other flagged rows for review. Do not assume eligibility timestamps are request submission times or must occur in a particular order.

`qc_source_row` identifies the 1-based source data record (excluding the header). Additional input columns are retained. The workflow does not automatically edit note text or interpret undocumented sentinel values.

In [3]:
tables, manifest = run(
    notes_path=NOTES_PATH,
    requests_path=REQUESTS_PATH,
    output_dir=OUTPUT_DIR,
    cutoff=CUTOFF,
    snapshot_label=SNAPSHOT_LABEL,
    preview_rows=PREVIEW_ROWS,
)

Completed full agreed files. Outputs: /Users/peytontom/dso576/misinformation-analysis/phase2-peyton/outputs
 dataset  input_rows  output_rows  exact_duplicate_rows_removed  conflicting_key_rows  original_columns  added_columns
   notes       15430        15430                             0                     0                24             25
requests      499888       499888                             0                     0                 7             20


## 4. Review the full cleaning report

Verify that input rows equal output rows plus exact duplicate removals. The script checks that original source fields survive saving/reloading and that the source hashes remain unchanged. Compare missingness before and after; counts may fall when identical rows are removed.

In [4]:
display(tables['cleaning_report'])
display(tables['column_profile'])
display(tables['quality_flags'])
print('Scope:', manifest['scope'])
print('Checks:', manifest['checks'])

,dataset,input_rows,output_rows,exact_duplicate_rows_removed,conflicting_key_rows,original_columns,added_columns
0,notes,15430,15430,0,0,24,25
1,requests,499888,499888,0,0,7,20


,dataset,stage,column,dtype,rows,empty,whitespace_only,blank_total
0,notes,before,noteId,string,15430,0,0,0
1,notes,before,noteAuthorParticipantId,string,15430,0,0,0
2,notes,before,createdAtMillis,string,15430,0,0,0
3,notes,before,tweetId,string,15430,0,0,0
4,notes,before,classification,string,15430,0,0,0
...,...,...,...,...,...,...,...,...
57,requests,after,apiSmallFeedEligibleAtMillis,string,499888,0,0,0
58,requests,after,apiLargeFeedEligibleAtMillis,string,499888,0,0,0
59,requests,after,apiXlFeedEligibleAtMillis,string,499888,0,0,0
60,requests,after,sourceLinks,string,499888,379877,0,379877


,dataset,flag,count
0,notes,qc_tweetId_missing,0
1,notes,qc_tweetId_invalid,0
2,notes,qc_noteId_missing,0
3,notes,qc_noteId_invalid,0
4,notes,qc_key_conflict,0
5,notes,qc_createdAtMillis_missing,0
6,notes,qc_createdAtMillis_invalid,0
7,notes,qc_createdAtMillis_nonpositive,0
8,notes,qc_misleadingOther_unexpected_binary,0
9,notes,qc_misleadingFactualError_unexpected_binary,0


Scope: full agreed files
Checks: {'row_counts_reconciled': True, 'source_hashes_unchanged': True, 'saved_original_fields_verified': True}


## 5. Check tweet relationships

`both` means the tweet appears in both agreed files; `notes_only` and `requests_only` are unmatched membership categories, not errors. Invalid or blank tweet IDs are excluded from this comparison and counted in the quality report.

Notes can contain multiple notes per tweet. Repeated request keys are retained and reported; do not perform a many-to-one join until those conflicts are resolved. Snapshot coverage can explain unmatched tweets.

In [5]:
display(tables['relationship_summary'])
display(tables['tweet_relationships'].head(20))
display(tables['tweet_relationships'].query('notes_rows > 1 or request_rows > 1').head(20))

,relationship,distinct_tweets
0,both,4254
1,notes_only,5432
2,requests_only,495634


,tweetId,notes_rows,request_rows,relationship
0,1253353708961902596,1,0,notes_only
1,1286632257076289536,1,1,both
2,1412888859885592581,1,1,both
3,1499398710446358535,3,1,both
4,1647579423602794496,1,0,notes_only
5,1670899448086446080,1,0,notes_only
6,1695829575060623534,1,1,both
7,1712811464396878322,1,0,notes_only
8,1717420732949430698,1,0,notes_only
9,1733814817943630289,1,1,both


,tweetId,notes_rows,request_rows,relationship
3,1499398710446358535,3,1,both
15,1783830499317960758,2,0,notes_only
27,1845440230427590726,3,0,notes_only
32,1858043968870654139,2,1,both
36,1881408246067744887,2,1,both
43,1905621521155969355,2,1,both
53,1928677825097380007,2,0,notes_only
56,1934909404631228713,2,0,notes_only
60,1942066201145889239,2,0,notes_only
61,1943029190023884914,3,0,notes_only


## 6. Review flagged records

The following displays at most 20 examples per table. Use the complete CSV flags for a comprehensive review. No further deletion is justified solely by a flag.

In [6]:
for name in ['notes_clean', 'note_requests_clean']:
    frame = tables[name]
    flags = [c for c in frame if c.startswith('qc_') and c != 'qc_source_row']
    print(name)
    display(frame.loc[frame[flags].any(axis=1)].head(20))
print('Limitations:')
for limitation in manifest['limitations']:
    print('-', limitation)

notes_clean


,noteId,noteAuthorParticipantId,createdAtMillis,tweetId,classification,believable,harmful,validationDifficulty,misleadingOther,misleadingFactualError,...,qc_misleadingUnverifiedClaimAsFact_unexpected_binary,qc_misleadingSatire_unexpected_binary,qc_notMisleadingOther_unexpected_binary,qc_notMisleadingFactuallyCorrect_unexpected_binary,qc_notMisleadingOutdatedButNotWhenWritten_unexpected_binary,qc_notMisleadingClearlySatire_unexpected_binary,qc_notMisleadingPersonalOpinion_unexpected_binary,qc_trustworthySources_unexpected_binary,qc_isMediaNote_unexpected_binary,qc_isCollaborativeNote_unexpected_binary


note_requests_clean


,tweetId,noteRequestFeedEligibleAtMillis,apiSmallFeedEligibleAtMillis,apiLargeFeedEligibleAtMillis,apiXlFeedEligibleAtMillis,sourceLinks,suggestions,qc_source_row,qc_tweetId_missing,qc_tweetId_invalid,...,qc_apiSmallFeedEligibleAtMillis_invalid,qc_apiSmallFeedEligibleAtMillis_nonpositive,apiLargeFeedEligibleAtMillis_utc,qc_apiLargeFeedEligibleAtMillis_missing,qc_apiLargeFeedEligibleAtMillis_invalid,qc_apiLargeFeedEligibleAtMillis_nonpositive,apiXlFeedEligibleAtMillis_utc,qc_apiXlFeedEligibleAtMillis_missing,qc_apiXlFeedEligibleAtMillis_invalid,qc_apiXlFeedEligibleAtMillis_nonpositive
0,1463732880,-1,-1,-1,1787257421863,,[],1,False,False,...,False,True,NaT,False,False,True,2026-08-20 20:23:41.863000+00:00,False,False,False
1,1711529307,-1,-1,-1,1788047004994,,[],2,False,False,...,False,True,NaT,False,False,True,2026-08-29 23:43:24.994000+00:00,False,False,False
2,2095550192,-1,-1,-1,1790712587377,,"[{""suggestion_id"":2105026835134218452,""suggest...",3,False,False,...,False,True,NaT,False,False,True,2026-09-29 20:09:47.377000+00:00,False,False,False
3,4176682202,-1,-1,-1,1786803953724,,"[{""suggestion_id"":2088632163939832178,""suggest...",4,False,False,...,False,True,NaT,False,False,True,2026-08-15 14:25:53.724000+00:00,False,False,False
4,6898415438,-1,-1,-1,1790006433504,,[],5,False,False,...,False,True,NaT,False,False,True,2026-09-21 16:00:33.504000+00:00,False,False,False
5,9016888221,-1,-1,1789819678503,1789819678503,https://x.com/ceomuskprizee/status/9016888221,"[{""suggestion_id"":2101281913109397857,""suggest...",6,False,False,...,False,True,2026-09-19 12:07:58.503000+00:00,False,False,False,2026-09-19 12:07:58.503000+00:00,False,False,False
6,11195978639,-1,-1,-1,1790271548683,,[],7,False,False,...,False,True,NaT,False,False,True,2026-09-24 17:39:08.683000+00:00,False,False,False
7,15247548894,-1,-1,-1,1791790359362,,[],8,False,False,...,False,True,NaT,False,False,True,2026-10-12 07:32:39.362000+00:00,False,False,False
8,15821970922807296,-1,-1,-1,1791645405858,,"[{""suggestion_id"":2108939205275705739,""suggest...",9,False,False,...,False,True,NaT,False,False,True,2026-10-10 15:16:45.858000+00:00,False,False,False
9,57293389854556160,-1,-1,-1,1790246843877,,[],10,False,False,...,False,True,NaT,False,False,True,2026-09-24 10:47:23.877000+00:00,False,False,False


Limitations:
- Agreed files do not establish platform-wide coverage.
- Missing eligibility dates and nonpositive values are flagged, not assigned undocumented meanings.
- No eligibility ordering or request-to-note causal/timing assumptions are applied.
- Future-date validation is unavailable without an explicit snapshot cutoff.
- Snapshot date/version unknown.


## 7. Write your findings and submit

After a successful **full-file** run, replace the prompts below with the actual report values and your review:

- Source snapshot/download information and cutoff:
- Notes: input rows, exact duplicates removed, output rows:
- Note Request: input rows, exact duplicates removed, output rows:
- Missing identifiers, conflicting keys, and timestamp concerns:
- Distinct tweets in both files / Notes only / Note Request only:
- What was changed, what was retained, and why:
- Remaining limitations and any follow-up with the team:

Submit this saved, executed notebook, `clean_notes_requests.py`, `notes_clean.csv`, `note_requests_clean.csv`, and the small audit reports/manifest as required by the team. Full outputs are in `outputs/`; preview outputs are in `outputs/preview/`. Do not submit preview results as the full cleaning result.

The pipeline loads both files into memory. If the real files exceed available RAM, adapt the workflow to chunked processing with global duplicate and relationship checks before attempting a final run.

## Completed full-file results

The notebook ran successfully on all rows of the user-confirmed `notes-00001.tsv` and `batSignals-00000.tsv`.

| Check | Notes | Note Request |
|---|---:|---:|
| Input rows | 15,430 | 499,888 |
| Cleaned rows | 15,430 | 499,888 |
| Exact duplicate rows removed | 0 | 0 |
| Conflicting primary-key rows | 0 | 0 |
| Missing or invalid tweet IDs | 0 | 0 |

All note IDs were present and valid under the positive-decimal format check. All Notes creation timestamps converted successfully and were positive. No nonblank checked binary indicators fell outside 0/1.

**Missing fields:** `believable`, `harmful`, and `validationDifficulty` are empty in all 15,430 Notes rows. `sourceLinks` is empty in 379,877 request rows (76.0%). These fields were retained without imputation. Other original fields contain no empty or whitespace-only values. Nonempty structured text such as `[]` is not counted as a blank string.

**Eligibility timestamps:** the following counts are nonpositive source values. Their originals are retained, their UTC dates are missing, and flags identify them. All observed nonpositive values are `-1`; no undocumented interpretation is assigned to that sentinel.

| Field | Nonpositive rows |
|---|---:|
| noteRequestFeedEligibleAtMillis | 470,723 |
| apiSmallFeedEligibleAtMillis | 470,395 |
| apiLargeFeedEligibleAtMillis | 326,414 |
| apiXlFeedEligibleAtMillis | 200 |

There were no blank or malformed eligibility timestamp strings. That does not establish that every timestamp represents an actual eligibility event.

**Tweet relationships:** 4,254 distinct tweets appear in both files, 5,432 only in Notes, and 495,634 only in Note Request. Notes contains 9,686 distinct tweets across 15,430 notes; repeated tweet IDs were retained. Unmatched tweets were retained and no raw-table merge was performed.

**Changes made:** preserved every original field and row; added UTC dates, source-row references, and quality flags. Both cleaned CSVs were reloaded and their original fields compared with the inputs. Input source hashes remained unchanged. Synthetic edge-case checks and execution of every notebook cell passed.

**Limitations:** the actual source snapshot/download date and observation cutoff have not been confirmed. Future-date validation is therefore not claimed. These results describe the two agreed files, not all Community Notes data. The original mention of `notes-00002.tsv` was replaced by `notes-00001.tsv` with the user's explicit confirmation. Review the records and these findings before submitting them as your work.
